<a href="https://colab.research.google.com/github/shoukk8-afk/MNIST.2/blob/main/MNIST%E2%91%A1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms

In [ ]:
#transformsを構成
train_transform = transforms.Compose([
    # ±10度の範囲でランダムに回転, fill=0で足りなくなった部分を黒で埋める
    transforms.RandomRotation(10, fill=0),
    # 0.8倍〜1.2倍の範囲でランダムに拡大縮小
    transforms.RandomAffine(degrees=0, scale=(0.8, 1.2)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

In [ ]:
train_dataset = datasets.MNIST(root='/content/', train=True, download=True, transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_dataset = datasets.MNIST(root='/content/', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=True)

In [ ]:
#CNNクラスの作成
#画像サイズを変えたくないためpadding=1を指定
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.fc1 = nn.Linear(1568, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        out = self.pool(F.relu(self.conv1(x)))
        out = self.pool(F.relu(self.conv2(out)))

        out = out.view(-1, 1568)

        out = F.relu(self.fc1(out))
        out = self.fc2(out)
        return out

In [ ]:
model = CNN()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

#クラス分類のためクロスエントロピー誤差を採用
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 1e-2)

In [ ]:
#訓練ループの作成
#損失の平均値を記録
def training_loop(n_epochs, optimizer, model, loss_fn, train_loader, test_loader):
    for epoch in range(1, n_epochs + 1):
        total = 0
        correct = 0
        loss_train =  0.0
        model.train()
        for imgs, labels in train_loader:
            #imgsとlabelsをGPUに移す
            imgs = imgs.to(device=device)
            labels = labels.to(device=device)

            optimizer.zero_grad()
            outputs = model(imgs)

            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()

            #1回のループの損失を加算して訓練データの1エボックの損失の合計を出す
            loss_train += loss.item()
        with torch.no_grad():
            model.eval()
            loss_val = 0.0
            for imgs, labels in test_loader:
                imgs = imgs.to(device=device)
                labels = labels.to(device=device)
                outputs = model(imgs)
                loss = loss_fn(outputs, labels)

                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

                #テストデータも同様に1エボックごとの損失の合計を出す
                loss_val += loss.item()
            accuracy = correct / total

        print(f"Epoch: {epoch}, Loss: {loss_train / len(train_loader)}")
        print(f"Accuracy: {accuracy}, Valloss: {loss_val / len(test_loader)}")

In [ ]:
training_loop(10, optimizer, model, loss_fn, train_loader, test_loader)

In [ ]:
#DataLoaderから取り出す
dataiter = iter(test_loader)
images, labels = next(dataiter)

model.eval()
with torch.no_grad():
    outputs = model(images.to(device))
    _, predicted = torch.max(outputs, 1)

# テストデータの結果の可視化
fig = plt.figure(figsize=(10, 4))
for idx in range(10): # 最初の10枚を表示
    ax = fig.add_subplot(2, 5, idx+1, xticks=[], yticks=[])
    img = images[idx].numpy().squeeze() # チャネル次元を消す
    ax.imshow(img, cmap='gray')
    # タイトルの色：正解なら黒、不正解なら赤
    color = "black" if predicted[idx].item() == labels[idx].item() else "red"
    ax.set_title(f"Pred: {predicted[idx].item()}\n(True: {labels[idx].item()})", color=color)
plt.tight_layout()
plt.show()

In [ ]:
# 重みの保存
save_path = "mnist_cnn.pth"
torch.save(model.state_dict(), save_path)
print(f"Model saved to {save_path}")